# 02 — Model Evaluation

Back-tests the forecasters with a time-based holdout and inspects the stored
`ml_forecast` outputs. Run `python scripts/run_pipeline.py` first.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import duckdb, pandas as pd, numpy as np
from src.config import settings
con = duckdb.connect(str(settings.warehouse_path), read_only=True)
fact = con.execute('SELECT * FROM hospital_activity_fact').df()
fact['date_key'] = pd.to_datetime(fact['date_key'])
fact.shape

## Time-based back-test: bed occupancy (Prophet + XGBoost hybrid)
Train on all but the last 30 days, forecast, compare to actuals.

In [ ]:
from src.models.bed_occupancy import BedOccupancyForecaster
cutoff = fact['date_key'].max() - pd.Timedelta(days=30)
train = fact[fact['date_key'] <= cutoff]
actual = (fact[fact['date_key'] > cutoff]
          .groupby('date_key')['bed_occupancy_pct'].mean())
m = BedOccupancyForecaster(horizon_days=30)
m.fit(train, target='bed_occupancy_pct')
fc = m.forecast().set_index('date_key')['yhat']
fc.index = pd.to_datetime(fc.index)
joined = pd.concat([actual.rename('actual'), fc.rename('forecast')], axis=1).dropna()
print(BedOccupancyForecaster.evaluate(joined['actual'], joined['forecast']))
joined.plot(figsize=(10,3), title='Bed occupancy: 30-day back-test')

## Stored forecasts by target & horizon

In [ ]:
con.execute('''SELECT target, horizon_days, model, COUNT(*) n,
                      ROUND(AVG(yhat),2) avg_yhat
               FROM ml_forecast GROUP BY 1,2,3 ORDER BY 1,2''').df()

## Waiting-time forecast horizon comparison (national mean)

In [ ]:
con.execute('''SELECT horizon_days, ROUND(AVG(yhat),1) AS mean_pred_wait_days
               FROM ml_forecast WHERE target='waiting_time'
               GROUP BY 1 ORDER BY 1''').df()

## Risk score distribution

In [ ]:
con.execute('''SELECT classification, COUNT(*) n, ROUND(AVG(score),3) avg_score
               FROM risk_score GROUP BY 1 ORDER BY avg_score DESC''').df()

In [ ]:
con.close()